<a href="https://colab.research.google.com/github/febrianucup/PCVK-CItra-Vision-Computer/blob/main/Week6_Muhammad_Dwi_Febrianipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2 as cv
import math
from google.colab.patches import cv2_imshow
from PIL import Image as im

In [ ]:
def convolution2d(image, kernel, stride, padding):
  k_height, k_width = kernel.shape

  if padding > 0:
      padded_image = np.pad(image, ((padding, padding), (padding, padding)), mode='constant', constant_values=0)
  else:
      padded_image = image.copy()

  img_height, img_width = padded_image.shape

  # Menghitung dimensi output
  out_height = int((img_height - k_height) / stride) + 1
  out_width = int((img_width - k_width) / stride) + 1

  output = np.zeros((out_height, out_width), dtype=np.float32)

  # Proses konvolusi
  for y in range(0, out_height):
      for x in range(0, out_width):
          # Mengambil region of interest (ROI)
          y_start = y * stride
          x_start = x * stride
          region = padded_image[y_start:y_start+k_height, x_start:x_start+k_width]

          # Perkalian elemen dan penjumlahan
          output[y, x] = np.sum(region * kernel)

  # Clipping agar nilai piksel tetap berada pada rentang [0, 255]
  output = np.clip(output, 0, 255).astype(np.uint8)
  return output

In [ ]:
img = cv.imread('/content/drive/MyDrive/PCVK/Images/mandrill.tiff')
img_gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
cv2_imshow(img_gray)

**Kernel Sharpen**

In [ ]:
kernel_sharpen = np.array([[0, -1, 0],
                           [-1, 5, -1],
                           [0, -1, 0]])
convolution2d(img_gray, kernel_sharpen, 1, 2)

**Operasi Emboss**

In [ ]:
kernel_emboss = np.array([[-2, -1, 0],
                          [-1, 1, 1],
                          [0, 1, 2]])
convolution2d(img_gray, kernel_emboss, 1, 2)

**Left Sobel Edge Detection**

In [ ]:
kernel_LSED = np.array([[1, 0, -1],
                        [2, 0, -2],
                        [1, 0, -1]])
convolution2d(img_gray, kernel_LSED, 1, 2)

**Canny Edge Detection**

In [ ]:
kernel_CED = np.array([[-1, -1, -1],
                       [-1, 8, -1],
                       [-1, -1, -1]])
convolution2d(img_gray, kernel_CED, 1, 2)

**21x21 Gaussian Blur**

In [ ]:
kernel_size = 21
sigma=math.sqrt(kernel_size)
gaussian_kernel = cv.getGaussianKernel(kernel_size, sigma)
gauss_kernel = gaussian_kernel @gaussian_kernel.transpose()

convolution2d(img_gray, gauss_kernel, 1, 2)

**E. FILTER LIBRARY DAN FILTER MODERN**

Percobaan 1:

In [ ]:
# Fungsi tampil berdampingan
def show_side_by_side(images, titles, figsize=(15, 5)):
    plt.figure(figsize=figsize)
    for i, (img, title) in enumerate(zip(images, titles)):
        if len(img.shape) == 2:  # grayscale
            plt.subplot(1, len(images), i + 1)
            plt.imshow(img, cmap="gray")
        else:  # color
            plt.subplot(1, len(images), i + 1)
            plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
        plt.title(title)
        plt.axis("off")
    plt.show()


imgLena = cv.imread("/content/drive/MyDrive/PCVK/Images/lena.jpg")
imgLena_gray = cv.cvtColor(imgLena, cv.COLOR_BGR2GRAY)

blur = cv.GaussianBlur(imgLena, (7, 7), 1)
edges = cv.Canny(cv.cvtColor(imgLena, cv.COLOR_BGR2GRAY), 100, 200)
sharpen_kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
sharpened = cv.filter2D(imgLena, -1, sharpen_kernel)
show_side_by_side([imgLena, blur, sharpened, edges],
                  ["Asli", "Gaussian Blur", "Sharpened", "Canny Edge Detection"])

Percobaan 2:

In [ ]:
# Filter Modern dari OpenCV
# Bilateral Filter (edge-preserving)
bilateral = cv.bilateralFilter(imgLena, 50, 100, 100)

# Edge Preserving Filter (alternatif Guided Filter)
edge_preserve = cv.edgePreservingFilter(imgLena, flags=1, sigma_s=100, sigma_r=0.9)

show_side_by_side(
    [imgLena, bilateral, edge_preserve],
    ["Asli", "Bilateral Filter", "Edge Preserving Filter"])

Percobaan 3:

In [ ]:
# Filter Feature Map yang digunakan pada CNN, Lakukan running code bagian ini beberapa kali dan perhatikan hasilnya
import torch
import torch.nn as nn


class SimpleCNN(nn.Module):

    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 4, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        return self.conv1(x)


model = SimpleCNN()

# Ubah gambar ke tensor
img_tensor = (torch.tensor(imgLena_gray, dtype=torch.float32).unsqueeze(0).unsqueeze(0)/255.0)

# Hasil CNN
with torch.no_grad():
    features = model(img_tensor)

# Visualisasi feature maps
feature_maps = [features[0, i].numpy() for i in range(features.shape[1])]
show_side_by_side([imgLena_gray] + feature_maps, ["Asli (Gray)"] + [f"Feature {i+1}" for i in range(len(feature_maps))])

Percobaan 4:

In [ ]:
# ====================
# 1. Beauty Filter
# ====================
# Step 1: Smoothing kulit dengan bilateral filter
smooth = cv.bilateralFilter(imgLena, d=15, sigmaColor=75, sigmaSpace=75)

# Step 2: Unsharp masking (pertajam mata/bibir)
gaussian = cv.GaussianBlur(smooth, (0, 0), 3)
sharpened = cv.addWeighted(smooth, 1.5, gaussian, -0.5, 0)

# Step 3: Brightness & contrast
alpha = 1.2  # contrast
beta = 15  # brightness
beauty = cv.convertScaleAbs(sharpened, alpha=alpha, beta=beta)

# ====================
# 2. Old/Vintage Filter
# ====================
# Step 1: Sepia tone
sepia_kernel = np.array([[0.272, 0.534, 0.131], [0.349, 0.686, 0.168], [0.393, 0.769, 0.189]])
sepia = cv.transform(imgLena, sepia_kernel)
sepia = np.clip(sepia, 0, 255).astype(np.uint8)

# Step 2: Vignette
rows, cols = imgLena.shape[:2]
kernel_x = cv.getGaussianKernel(cols, cols * 0.6)
kernel_y = cv.getGaussianKernel(rows, rows * 0.6)
kernel = kernel_y * kernel_x.T
mask = kernel / kernel.max()
vignette = np.copy(sepia)
for i in range(3):
    vignette[:, :, i] = vignette[:, :, i] * mask

# Step 3: Noise/Grain
noise = np.random.normal(0, 15, vignette.shape).astype(np.int16)
old_img = np.clip(vignette.astype(np.int16) + noise, 0, 255).astype(np.uint8)

show_side_by_side([imgLena, beauty, old_img], ['Asli', 'Beauty Filter', 'Old/Vintage Filter'])

Percobaan 5:

In [ ]:
# Filter Anime / Cartoon
# Step 1: Edge detection (pakai median blur dulu agar gambar menjadi lebih halus)
gray = cv.cvtColor(imgLena, cv.COLOR_BGR2GRAY)
gray_blur = cv.medianBlur(gray, 7)
edges = cv.adaptiveThreshold(gray_blur, 255, cv.ADAPTIVE_THRESH_MEAN_C, cv.THRESH_BINARY, 9, 9,)

# Step 2: Bilateral filter untuk smoothing warna
color = cv.bilateralFilter(imgLena, d=9, sigmaColor=200, sigmaSpace=200)

# Step 3: Gabungkan (cartoonize)
cartoon = cv.bitwise_and(color, color, mask=edges)

# Tampilkan
show_side_by_side([imgLena, cartoon], ["Asli", "Cartoon Filter"])

Percobaan 6:

In [ ]:
# Night Filter
imgJawatan = cv.imread("/content/drive/MyDrive/PCVK/Images/djawatan.jpg")
imgJawatan_gray = cv.cvtColor(imgJawatan, cv.COLOR_BGR2GRAY)

# Step 1: Gelapkan (contrast turun, brightness negatif)
night = cv.convertScaleAbs(imgJawatan, alpha=0.6, beta=-40)

# Step 2: Tambah bias biru
blue_tint = np.full_like(night, (50, 0, 100))  # BGR
night = cv.addWeighted(night, 0.8, blue_tint, 0.2, 0)

# Step 3: Efek glow di area terang dengan filter2D (blur kernel)
kernel = np.ones((15, 15), np.float32) / 225
glow = cv.filter2D(night, -1, kernel)

# Kombinasikan asli + glow
night_glow = cv.addWeighted(night, 0.7, glow, 0.3, 0)

show_side_by_side([imgJawatan, night, night_glow],
                  ["Asli", "Night Filter", "Night + Glow (filter2D)"])

Percobaan 7:

In [ ]:
# Filter Suasana pagi dan Kabut
# ===========================
# Step 1: Kurangi kontras & cerahkan
# ===========================
alpha = 0.9  # contrast
beta = 20  # brightness
soft = cv.convertScaleAbs(imgJawatan, alpha=alpha, beta=beta)

# ===========================
# Step 2: Tambahkan warm tone (kemerahan / oranye)
# ===========================
warm_tint = np.full_like(soft, (40, 70, 120))  # BGR
pagi = cv.addWeighted(soft, 0.8, warm_tint, 0.2, 0)

# ===========================
# Step 3: Tambahkan haze (kabut tipis) dengan filter2D
# ===========================
# Kernel blur Gaussian-like untuk menciptakan efek kabut
kernel = cv.getGaussianKernel(3, 3)
kernel = kernel @ kernel.T  # jadikan 2D kernel
kabut = cv.filter2D(pagi, -1, kernel)

# tambah layer putih untuk kabut lebih nyata
white_layer = np.full_like(pagi, 255)
kabut = cv.addWeighted(kabut, 0.7, white_layer, 0.3, 0)

show_side_by_side([imgJawatan, pagi, kabut], ["Asli", "Pagi", "Pagi + Kabut"])

# **Tugas Praktikum**
1. **Tugas Analisis: Perbandingan Mean Filter (Blur Biasa) vs. Median Filter pada Noise “Salt and Pepper”**
  
  - a. Tambahkan noise bintik hitam-putih (Salt and Pepper) ke sebuah citra keabuan (grayscale).
  - b. Terapkan Mean Filter(3 x 3) dan Median Filter (3 x 3).Pengolahan Citra dan
  - c. Amati dan analisis secara visual mengapa Median Filter jauh lebih bersih dalam menghapus noise tersebut dibanding Mean Filter.

**a. Tambahkan noise bintik hitam-putih (Salt and Pepper) ke sebuah citra keabuan (grayscale).**

In [ ]:
def add_salt_and_pepper_noise(image, salt_prob, pepper_prob):
    noisy_image = image.copy()
    # Salt noise (putih)
    num_salt = np.ceil(salt_prob * image.size)
    coords = [np.random.randint(0, i - 1, int(num_salt)) for i in image.shape]
    noisy_image[tuple(coords)] = 255

    # Pepper noise (hitam)
    num_pepper = np.ceil(pepper_prob * image.size)
    coords = [np.random.randint(0, i - 1, int(num_pepper)) for i in image.shape]
    noisy_image[tuple(coords)] = 0
    return noisy_image

noisy_img = add_salt_and_pepper_noise(imgJawatan_gray, salt_prob=0.02, pepper_prob=0.02)
cv2_imshow(noisy_img)

**b. Terapkan Mean Filter(3 x 3) dan Median Filter (3 x 3).Pengolahan Citra**

In [ ]:
mean_filtered = cv.blur(noisy_img, (3, 3))
median_filtered = cv.medianBlur(noisy_img, 3)
show_side_by_side([mean_filtered, median_filtered],
                  ["Mean Filter", "Median Filter"])

**c. Amati dan analisis secara visual mengapa Median Filter jauh lebih bersih dalam menghapus noise tersebut dibanding Mean Filter.**

- Mean Filter (Rata-rata) menghitung rata-rata nilai piksel di area $3\times3$. Nilai noise ekstrem (seperti 0/hitam murni dan 255/putih murni) akan ikut terhitung ke dalam rata-rata, sehingga nilai bintik noise disebarkan ke piksel sekitarnya. Akibatnya, noise tidak hilang melainkan menjadi buram (blurry noise spot). <br>

- Median Filter mengurutkan seluruh piksel dalam jendela $3\times3$ lalu mengambil nilai tengahnya (median). Karena nilai salt and pepper selalu berada di ujung terluar (0 paling bawah atau 255 paling atas), nilai tersebut hampir pasti tidak akan terpilih sebagai nilai median. Efeknya, noise bintik hilang total tanpa mengaburkan detail utama citra.

**2. Tugas Evaluasi: Pemilihan Ukuran Kernel terbaik untuk Menajamkan Citra (Sharpening)**
- a. Buat kernel penajaman (Sharpening Kernel) ukuran 3 x 3 dan 5 x 5.
- b. Terapkan pada citra yang sedikit buram (blurry).
- c. Evaluasi hasilnya berdasarkan aspek Kejelasan Tepi vs. Kadar Noise/Noise Distortion yang dihasilkan. Tentukan kernel mana yang paling optimal untuk digunakan sehari-hari.

**a. Buat kernel penajaman (Sharpening Kernel) ukuran 3 x 3 dan 5 x 5.**

In [ ]:
kernel_sharpen = np.array([[0, -1, 0],
                           [-1, 5, -1],
                           [0, -1, 0]])
kernel_3x3 = convolution2d(imgJawatan_gray, kernel_sharpen, 1, 2)

kernel_sharpen_5x5 = np.array([[-1, -1, -1, -1, -1],
                               [-1, -1, -1, -1, -1],
                               [-1, -1, 25, -1, -1],
                               [-1, -1, -1, -1, -1],
                              [-1, -1, -1, -1, -1]])
kernel_5x5 = convolution2d(imgJawatan_gray, kernel_sharpen_5x5, 1, 2)
show_side_by_side([kernel_3x3, kernel_5x5],
                  ["Kernel 3x3", "Kernel 5x5"])

**b. Terapkan pada citra yang sedikit buram (blurry).**

In [ ]:
blurry_img = cv.GaussianBlur(imgJawatan, (7, 7), 2)
show_side_by_side([imgJawatan, blurry_img],
                  ["Asli", "Blurry"])

**c. Evaluasi hasilnya berdasarkan aspek Kejelasan Tepi vs. Kadar Noise/Noise Distortion yang dihasilkan. Tentukan kernel mana yang paling optimal untuk digunakan sehari-hari.**

- **Kernel 3 X 3:** Meningkatkan ketajaman batas/garis secara pas dan alami Akan tetapi Noise Distortion yang dihasilkan sangat rendah, menjaga kontinuitas tekstur tanpa memunculkan artefak tajam berlebihan.
- **Kernel 5 X 5:** Menghasilkan penajaman yang sangat agresif (kontras tepi terlalu tinggi/over-sharpened). Akan tetapi Noise Distortion yang dihasikan tinggi.
- **Kesimpulan:** Kernel 3 X *3* paling optimal untuk penggunaan sehari-hari karena memberikan keseimbangan terbaik antara peningkatan ketajaman tepi dan minimnya distorsi noise.

**3. Tugas Kreasi: Filter Kartun Sederhana**
<br>a. Buatlah sebuah fungsi kustom bernama kartun(image) yang menggabungkan:
* Penghalusan warna menggunakan **Bilateral Filter** agar area warna menjadi rata.
* Deteksi garis tepi menggunakan **Adaptive Thresholding** atau **Canny Edge**.
* Penggabungan (kombinasi) keduanya menggunakan operasi logika/perkalian piksel
<br>b. Tampilkan hasil citra asli dengan citra efek kartun bersebelahan.

In [ ]:
def custom_cartoon_img(image):
  color = cv.bilateralFilter(image, d=9, sigmaColor=200, sigmaSpace=200)
  bilateral = cv.bilateralFilter(image, 50, 100, 100)
  gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
  gray_blur = cv.medianBlur(gray, 7)
  edges = cv.adaptiveThreshold(gray_blur, 255, cv.ADAPTIVE_THRESH_MEAN_C, cv.THRESH_BINARY, 9, 9,)
  cartoon = cv.bitwise_and(color, color, mask=edges)

  return cartoon

**b. Tampilkan hasil citra asli dengan citra efek kartun bersebelahan.**

In [ ]:
show_side_by_side([imgJawatan, custom_cartoon_img(imgJawatan)],
                  ["Asli", "Custom Cartoon"])